## Bước 5: Text Preprocessing
### 5.1 Đọc dữ liệu

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/depression_severity_no_leakage.csv')

print("Số dòng:", len(df))
print(df.head(3))

Số dòng: 3519
                                                text    label
0  He said he had not felt that way before, sugge...     mild
1  Hey there r/assistance, Not sure if this is th...  minimum
2  My mom then hit me with the newspaper and it s...  minimum


### 5.2 Cài đặt NLTK (cho lemmatization)

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to C:\Users\Thanh
[nltk_data]     Hue\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Thanh
[nltk_data]     Hue\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to C:\Users\Thanh
[nltk_data]     Hue\AppData\Roaming\nltk_data...


True

### 5.3 Hàm tiền xử lý — Nhánh A (LogReg, SVM)

In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))

# Giữ lại các từ phủ định — quan trọng với bài toán phát hiện trầm cảm
negation_words = {'not', 'no', 'nor', 'never', "n't", 'none', 'nothing',
                    "don't", "doesn't", "didn't", "isn't", "wasn't", "aren't",
                    "won't", "wouldn't", "can't", "couldn't", "shouldn't"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def preprocess_classical(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r"[^a-z\s']", ' ', text)     # giữ dấu ' để không phá "don't" thành "don t"
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 1]

    return ' '.join(words)

sample = df['text'].iloc[0]
print("Gốc:", sample[:150])
print("\nSau xử lý:", preprocess_classical(sample)[:150])

Gốc: He said he had not felt that way before, suggeted I go rest and so ..TRIGGER AHEAD IF YOUI'RE A HYPOCONDRIAC LIKE ME: i decide to look up "feelings of

Sau xử lý: said not felt way suggeted go rest trigger ahead youi're hypocondriac like decide look feeling doom hope maybe getting sucked rabbit hole ludicrous co


### 5.4 Áp dụng tiền xử lý Nhánh A cho toàn bộ dữ liệu

In [5]:
df['text_classical'] = df['text'].apply(preprocess_classical)

print("Đã xử lý xong toàn bộ", len(df), "dòng")
print("\nĐộ dài trung bình trước xử lý:", df['text'].str.split().str.len().mean().round(1), "từ")
print("Độ dài trung bình sau xử lý:  ", df['text_classical'].str.split().str.len().mean().round(1), "từ")

Đã xử lý xong toàn bộ 3519 dòng

Độ dài trung bình trước xử lý: 85.6 từ
Độ dài trung bình sau xử lý:   39.4 từ


### 5.5 Dữ liệu Nhánh C — giữ nguyên bản gốc


In [6]:
df['text_neural'] = df['text']

print("Nhánh C (BiLSTM, BERT) dùng cột 'text_neural' — giữ nguyên câu gốc")
print("Ví dụ:")
print(df['text_neural'].iloc[0][:150])

Nhánh C (BiLSTM, BERT) dùng cột 'text_neural' — giữ nguyên câu gốc
Ví dụ:
He said he had not felt that way before, suggeted I go rest and so ..TRIGGER AHEAD IF YOUI'RE A HYPOCONDRIAC LIKE ME: i decide to look up "feelings of


### 5.6 Tổng kết và lưu file

In [7]:
df_final = df[['text', 'label', 'text_classical', 'text_neural']].copy()

print("Các cột trong dataset cuối cùng:", list(df_final.columns))
print("Số dòng:", len(df_final))

df_final.to_csv('../data/processed/depression_severity_preprocessed.csv', index=False)
print("\nĐã lưu: data/processed/depression_severity_preprocessed.csv")

Các cột trong dataset cuối cùng: ['text', 'label', 'text_classical', 'text_neural']
Số dòng: 3519

Đã lưu: data/processed/depression_severity_preprocessed.csv
